In [1]:
import requests

In [2]:
from copy import deepcopy

In [3]:
from time import sleep

In [13]:
import json

In [4]:
URL="https://nomad-lab.eu/backend/api/v1/entries/query"

In [5]:
BASE_QUERY = {
  "owner": "visible",
  "query": {
    "results.method.simulation.program_name:any": [
      "FHI-aims"
    ],
    "quantities:any": [
      "run.program.version"
    ]
  },
  "pagination": {
    "order_by": "upload_create_time",
    "order": "desc",
    "page_size": 1
  },
  "required": {
    "include": [
      "entry_id",
    ]
  }
}

In [6]:
result = requests.post(URL, json=BASE_QUERY)

In [7]:
result.raise_for_status()

In [9]:
result.json()['pagination']['total']

1390051

In [10]:
def get_total_usage_per_quantity(quantity_path, base_url = URL, query=BASE_QUERY):
    query = deepcopy(query)
    query["query"]["quantities:any"] = [quantity_path]
    result = requests.post(URL, json=query)
    result.raise_for_status()
    return result.json()['pagination']['total']

In [11]:
# this is what we use in the Simulations App
all_simulations = get_total_usage_per_quantity("results.method.simulation.program_name")

In [12]:
all_simulations

1390210

In [19]:
with open("data/quantities_seed0.json", "r") as f_:
    quantities = json.load(f_)

In [26]:
quantity_usages = {key: {} for key in quantities.keys()}
for key, quants in quantities.items():
    print(f"{key} - {len(quants)} quantities")
    for idx, quant in enumerate(quants):
        total = get_total_usage_per_quantity(quant)
        quantity_usages[key][quant] = {"total":total, "relative":total/all_simulations}
        print(f"{(idx + 1)/len(quants) * 100:.3f}", end = "\r")
    break

metadata - 69 quantities
100.000

In [36]:
with open("data/quantities_seed0_usages.json", "w") as f_:
    json.dump(quantity_usages, f_)

In [27]:
quantity_usages

{'metadata': {'metadata.calc_hash': {'total': 774753,
   'relative': 0.5572920637889239},
  'metadata.calc_id': {'total': 774753, 'relative': 0.5572920637889239},
  'metadata.coauthor_groups': {'total': 116665,
   'relative': 0.0839189762697722},
  'metadata.coauthors': {'total': 1390210, 'relative': 1.0},
  'metadata.comment': {'total': 241812, 'relative': 0.17393918904338193},
  'metadata.datasets': {'total': 1390210, 'relative': 1.0},
  'metadata.domain': {'total': 1390051, 'relative': 0.9998856287898951},
  'metadata.embargo_length': {'total': 1390210, 'relative': 1.0},
  'metadata.entry_coauthors': {'total': 1390210, 'relative': 1.0},
  'metadata.entry_create_time': {'total': 1390210, 'relative': 1.0},
  'metadata.entry_hash': {'total': 615457, 'relative': 0.442707936211076},
  'metadata.entry_id': {'total': 615457, 'relative': 0.442707936211076},
  'metadata.entry_name': {'total': 155147, 'relative': 0.11159968637831695},
  'metadata.entry_timestamp': {'total': 153761,
   'relati

In [32]:
# thanks chatty
def to_markdown_tables(data: dict) -> str:
    lines = []

    for section, quantities in data.items():
        lines.extend([
            f"## {section}",
            "",
            "| Quantity | Total | Relative |",
            "| --- | ---: | ---: |",
        ])

        for quantity, stats in quantities.items():
            lines.append(
                f"| `{quantity}` | {stats['total']:,} | {stats['relative']:.2%} |"
            )

        lines.append("")

    return "\n".join(lines)

In [34]:
print(to_markdown_tables(quantity_usages))

## metadata

| Quantity | Total | Relative |
| --- | ---: | ---: |
| `metadata.calc_hash` | 774,753 | 55.73% |
| `metadata.calc_id` | 774,753 | 55.73% |
| `metadata.coauthor_groups` | 116,665 | 8.39% |
| `metadata.coauthors` | 1,390,210 | 100.00% |
| `metadata.comment` | 241,812 | 17.39% |
| `metadata.datasets` | 1,390,210 | 100.00% |
| `metadata.domain` | 1,390,051 | 99.99% |
| `metadata.embargo_length` | 1,390,210 | 100.00% |
| `metadata.entry_coauthors` | 1,390,210 | 100.00% |
| `metadata.entry_create_time` | 1,390,210 | 100.00% |
| `metadata.entry_hash` | 615,457 | 44.27% |
| `metadata.entry_id` | 615,457 | 44.27% |
| `metadata.entry_name` | 155,147 | 11.16% |
| `metadata.entry_timestamp` | 153,761 | 11.06% |
| `metadata.entry_timestamp.timestamp` | 153,761 | 11.06% |
| `metadata.entry_timestamp.token` | 153,761 | 11.06% |
| `metadata.entry_timestamp.token_seed` | 153,761 | 11.06% |
| `metadata.entry_timestamp.tsa_server` | 153,761 | 11.06% |
| `metadata.entry_type` | 155,163 | 11.